# §G — Served LoRA-export NLL gate (in-cluster, via cogflow)

The Phase-1 sibling of the §F `ntk_model` served notebook: confirm the **served** Phase-1 LoRA fine-tune reproduces the exported adapter's quality on a real KServe ISVC, driven end-to-end through **cogflow** from a Kubeflow notebook.

### Why this serves a *merged* model, not an attached adapter
The platform does **not** attach a LoRA adapter on the vLLM LLM serving path today — `POST /models-serving` with `lora_model_ids` alongside an `llm` `model_id` is rejected 422 (`model_register_service.py::_deploy_llm`, *"lora_model_ids is not supported on the LLM serving path"*), and the classical predictor route can't target an `llm` base. So to get a **served** LoRA number through cogflow, this notebook:
1. pulls the exported PEFT adapter (the artifact `controller_to_lora` produced) via cogflow,
2. **merges** it into the base with PEFT (`merge_and_unload`) — the served weights then *are* `W + BA`, i.e. exactly what stock `--enable-lora` would apply at request time,
3. logs the merged model as an MLflow artifact + registers it as an `llm` row via cogflow, and
4. serves it via the ordinary cogflow LLM path and scores served NLL.

Serving the merged model is behaviourally identical to serving base+adapter (LoRA is linear), so this is a faithful served test of the Phase-1 export. When the platform gains native LoRA attach, swap step 5's deploy for the commented `lora_model_ids` cell — the rest is unchanged.

**Metric / gate.** Same teacher-forced, completion-masked served NLL as §F (exact token ids over `/v1/completions`, `echo`+`logprobs`). The tight gate is **serving-correctness**: served ≈ the *local* §E `lora-export` NLL (`REFERENCE_NLL`) within 1% — did the pod stage + load the merged weights and preserve the math? The wider `hooks` reference (`REFERENCE_HOOKS_NLL`) is shown for context — the LoRA approximation itself sits a few % off the hook-attached controller.

Prereqs: run from a Kubeflow notebook with cogflow + Kubernetes access to your namespace; a base LLM `model_info` row; and `REFERENCE_NLL` computed by the local §E notebook (`ntk_lora_export_nll.ipynb`) on the **same** controller/adapter.

In [ ]:
# !pip install transformers peft requests cogflow

In [ ]:
# --- Config -------------------------------------------------------------
BASE = "Qwen/Qwen2.5-0.5B-Instruct"        # HF id of the base the adapter targets

# Identity of the rows. The LoRA row is the model_info(type='lora') the
# fine-tune pipeline registered (its id == the MLflow run holding the adapter).
BASE_MODEL_ID = "8f7318ab-6e4e-4e94-aeb9-6a61b6dd51e6"  # dev: Qwen2.5-0.5B-Instruct llm row (GET /models?type=llm)
LORA_MODEL_ID = None                         # dev: a model_info(type='lora') row id; None -> fine-tune below

# Eval + (optional) fine-tune datasets — registered cogflow JSONL datasets.
EVAL_DATASET_ID = "39030a5d-b36c-4f07-8895-667515fcaa14"  # GSM8K test split, 32 held-out rows
DATASET_ID = "a6693b90-b11a-4af1-a31b-51cd3c1a84e8"       # GSM8K train split (fine-tune option)
RUN_FINETUNE = False                         # True -> submit an export='lora' fine-tune to create LORA_MODEL_ID

# Names for the merged model + its ISVC.
MERGED_MODEL_NAME = "ntk-lora-gsm8k-merged"   # name of the registered llm row + served model name
MERGED_ISVC_NAME = "ntk-lora-gsm8k"           # ISVC name to deploy under
MERGED_DIR = "runs/_merged"                    # local dir the merged HF model is written to
MERGED_MODEL_ID = None                         # set by the register cell (== the merged model's MLflow run id)

# Cluster wiring — all auto-resolve from your user context; override only if needed.
NAMESPACE = None                             # None -> your user namespace (cogflow common.get_namespace())
COGAPI = None                                # None -> cogflow config.API_PATH
USER = None                                  # None -> cogflow common.get_current_user()
ENDPOINT = None                              # None -> resolve via cogflow
SERVED_MODEL_NAME = None                     # None -> read from /v1/models
API_KEY = None                               # only for external-ingress/cross-ns access; in-namespace needs none

# Gate: served (merged LoRA) vs the LOCAL §E lora-export arm on the same adapter.
REFERENCE_NLL = None                         # <-- set to the §E 'lora-export' NLL from ntk_lora_export_nll.ipynb
REFERENCE_HOOKS_NLL = 0.5886                  # §E 'reference' (hook-attached) arm, for context only
THRESHOLD = 0.01                             # |served-reference|/reference serving-correctness band
MAX_TOKENS = 0                               # 0 = echo only; auto-falls back to 1 if the server rejects 0

In [ ]:
import time
import requests

# Lazy-resolve CogAPI base + user from cogflow (idempotent; reused everywhere).
if COGAPI is None:
    from cogflow.config import config as _cog_config
    COGAPI = _cog_config.API_PATH
if USER is None:
    from cogflow.utils import common as _cog_common
    USER = _cog_common.get_current_user()
COGAPI = COGAPI.rstrip("/")
print("CogAPI:", COGAPI, "| user:", USER)

## 1. (optional) Fine-tune to create the LoRA adapter
Submit an `export='lora'` fine-tune over `BASE_MODEL_ID` + `DATASET_ID` (the pipeline runs `controller_to_lora` and registers a `model_info(type='lora')` row), then poll until the row appears — its id becomes `LORA_MODEL_ID`. Needs GPU/kfp. Skip (`RUN_FINETUNE=False`) if you already have a LoRA row.

In [ ]:
def list_lora_rows():
    """model_info(type='lora') rows for BASE_MODEL_ID (empty when none -> 404)."""
    r = requests.get(
        f"{COGAPI}/models",
        params={"type": "lora", "base_model_id": BASE_MODEL_ID},
        headers={"kubeflow-userid": USER}, timeout=30,
    )
    if r.status_code == 404:
        return []
    r.raise_for_status()
    return [m["id"] for m in r.json().get("data", [])]


if RUN_FINETUNE:
    before = set(list_lora_rows())
    body = {
        "base_model_id": BASE_MODEL_ID,
        "dataset_id": DATASET_ID,
        "output_name": MERGED_ISVC_NAME,
        "method": "ntk",
        "export": "lora",
        "hyperparams": {"gates": 5000, "max_log_gate": 0.05, "train_steps": 240, "lr": 5e-3},
    }
    r = requests.post(
        f"{COGAPI}/models/fine-tune", json=body,
        headers={"Content-Type": "application/json", "kubeflow-userid": USER}, timeout=120,
    )
    if r.status_code >= 400:
        raise RuntimeError(f"fine-tune failed [{r.status_code}]: {r.text}")
    print("fine-tune submitted:", r.json().get("data"))

    deadline = time.time() + 3600  # 1h
    while True:
        new = set(list_lora_rows()) - before
        if new:
            LORA_MODEL_ID = sorted(new)[-1]
            print("lora row registered:", LORA_MODEL_ID)
            break
        if time.time() > deadline:
            raise TimeoutError("fine-tune did not register a lora row within timeout")
        print("waiting for fine-tune to register the adapter ...")
        time.sleep(30)
else:
    print("RUN_FINETUNE=False -> using existing LORA_MODEL_ID:", LORA_MODEL_ID)

assert LORA_MODEL_ID, "set LORA_MODEL_ID (an existing lora row) or RUN_FINETUNE=True"

## 2. Pull the exported adapter via cogflow, and merge it into the base
The LoRA row's id is its MLflow run; the fine-tune component logged the adapter under the run's `adapter/` subpath. Download it through cogflow, merge with PEFT (`merge_and_unload`), and save a standalone HF model — the served weights are then exactly `W + BA`.

In [ ]:
from pathlib import Path
import cogflow.core.models as cogflow_models

# The adapter lives under <run artifact root>/adapter (artifact_subpath='adapter'
# in ntk_fine_tune_component). get_run_artifact_uri returns the run root.
adapter_uri = cogflow_models.get_run_artifact_uri(str(LORA_MODEL_ID)).rstrip("/") + "/adapter"
print("adapter artifact uri:", adapter_uri)

# cogflow configures the MLflow tracking + S3 env on import, so mlflow's
# artifact downloader resolves this URI with no extra credentials.
import mlflow  # noqa: E402  (cogflow re-exports/configures mlflow)
local_adapter = mlflow.artifacts.download_artifacts(artifact_uri=adapter_uri)
print("downloaded adapter to:", local_adapter)
print("adapter files:", sorted(p.name for p in Path(local_adapter).iterdir()))

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"[merge] loading base {BASE!r} + adapter {local_adapter}")
base_model = AutoModelForCausalLM.from_pretrained(BASE)
merged = PeftModel.from_pretrained(base_model, local_adapter).merge_and_unload()

merged_dir = Path(MERGED_DIR)
merged_dir.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(merged_dir))
# The tokenizer must ride along so vLLM can load the served dir standalone.
AutoTokenizer.from_pretrained(BASE).save_pretrained(str(merged_dir))
print("[merge] merged model written to:", merged_dir)
print("[merge] files:", sorted(p.name for p in merged_dir.iterdir()))

del base_model, merged  # free host RAM before serving

## 3. Register the merged model as an `llm` row (via cogflow)
Log the merged HF files at a run's artifact **root**, then register a `model_info(type='llm')` row whose id **is** that run id — so `POST /models-serving` with `model_id=<run>` resolves `storageUri` straight to the merged weights (`_deploy_llm` → `get_run_artifact_uri(model_id)`).

In [ ]:
with cogflow_models.start_run(run_name=f"merged-{MERGED_ISVC_NAME}") as run:
    MERGED_MODEL_ID = run.info.run_id
    cogflow_models.set_tag("type", "llm")
    cogflow_models.set_tag("lora_source_id", str(LORA_MODEL_ID))
    # Log at artifact root so <artifact_uri> itself is the loadable HF dir.
    cogflow_models.log_artifacts(str(merged_dir))
print("merged model logged under run:", MERGED_MODEL_ID)

# Register the llm catalog row with id == the run id (so serving resolves
# the artifact). /models/log accepts an explicit model_id.
r = requests.post(
    f"{COGAPI}/models/log",
    json={
        "model_id": MERGED_MODEL_ID,
        "model_name": MERGED_MODEL_NAME,
        "type": "llm",
        "hf_model_id": BASE,
        "description": f"NTK LoRA-export ({LORA_MODEL_ID}) merged into {BASE}",
    },
    headers={"Content-Type": "application/json", "kubeflow-userid": USER}, timeout=60,
)
if r.status_code >= 400:
    raise RuntimeError(f"/models/log failed [{r.status_code}]: {r.text}")
print("registered llm row:", r.json().get("data", {}).get("id", MERGED_MODEL_ID))

## 4. Deploy the merged model as an ISVC (via CogAPI) and wait for ready
Ordinary cogflow LLM deploy — `model_id` = the merged `llm` row. No `--quantization` (keep the served dtype clean for an apples-to-apples NLL). Idempotent: skips create if the ISVC exists.

> **When native LoRA attach lands**, replace the body with the commented `lora_model_ids` payload below and deploy the base directly — no merge/register needed.

In [ ]:
from cogflow import serving as cogflow_serving


def isvc_status(isvc_name, namespace=None):
    models = cogflow_serving.list_models(namespace=namespace, isvc_name=isvc_name)
    return models[0].get("status") if models else None


existing = isvc_status(MERGED_ISVC_NAME, NAMESPACE)
if existing is not None:
    print(f"ISVC {MERGED_ISVC_NAME!r} already exists (status={existing!r}); skipping create")
else:
    body = {
        "isvc_name": MERGED_ISVC_NAME,
        "model_id": MERGED_MODEL_ID,
        "served_model_name": MERGED_MODEL_NAME,
    }
    # --- When native LoRA attach is supported, use this instead of the
    #     merge+register path (skip sections 2-3 and set model_id=BASE_MODEL_ID):
    # body = {"isvc_name": MERGED_ISVC_NAME, "model_id": BASE_MODEL_ID,
    #         "lora_model_ids": [LORA_MODEL_ID]}   # currently 422 on the vLLM path
    r = requests.post(
        f"{COGAPI}/models-serving", json=body,
        headers={"Content-Type": "application/json", "kubeflow-userid": USER}, timeout=120,
    )
    if r.status_code >= 400:
        raise RuntimeError(f"deploy failed [{r.status_code}]: {r.text}")
    print("deploy accepted:", r.json())

deadline = time.time() + 900  # 15 min
while True:
    st = isvc_status(MERGED_ISVC_NAME, NAMESPACE)
    print("status:", st)
    if st == "ready":
        break
    if time.time() > deadline:
        raise TimeoutError(f"ISVC {MERGED_ISVC_NAME!r} not ready within timeout (last status={st!r})")
    time.sleep(15)

## 5. Resolve endpoint + served model name, and download the eval set (via cogflow)

In [ ]:
import zipfile
from cogflow import datasets as cogflow_datasets


def resolve_endpoint(isvc_name, namespace=None):
    models = cogflow_serving.list_models(namespace=namespace, isvc_name=isvc_name)
    if not models:
        raise RuntimeError(f"ISVC {isvc_name!r} not found in namespace {namespace!r}")
    url = models[0].get("served_model_url")
    if not url:
        raise RuntimeError(f"ISVC {isvc_name!r} has no served_model_url (status={models[0].get('status')!r})")
    return url.rstrip("/") + "/openai/v1"   # KServe HF runtime mounts OpenAI here


def discover_served_model_name(endpoint, api_key=None):
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    r = requests.get(endpoint.rstrip("/") + "/models", headers=headers, timeout=30)
    r.raise_for_status()
    data = r.json().get("data", [])
    if not data:
        raise RuntimeError("served /v1/models returned no models")
    return data[0]["id"]


def download_eval_jsonl(dataset_id, work_dir="runs/_eval"):
    """Download a cogflow dataset and return the local .jsonl (unwrap ZIP)."""
    work = Path(work_dir); work.mkdir(parents=True, exist_ok=True)
    downloaded = Path(cogflow_datasets.download_dataset(dataset_id, output_path=str(work)))
    if not zipfile.is_zipfile(downloaded):
        return str(downloaded)
    extracted = work / "extracted"; extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(downloaded) as zf:
        zf.extractall(extracted)
    jsonls = sorted(p for p in extracted.rglob("*") if p.is_file() and p.suffix.lower() == ".jsonl")
    if len(jsonls) != 1:
        raise RuntimeError(f"expected exactly one .jsonl in dataset {dataset_id}, found {jsonls}")
    return str(jsonls[0])


if ENDPOINT is None:
    ENDPOINT = resolve_endpoint(MERGED_ISVC_NAME, NAMESPACE)
    print("resolved endpoint :", ENDPOINT)
if SERVED_MODEL_NAME is None:
    SERVED_MODEL_NAME = discover_served_model_name(ENDPOINT, API_KEY)
print("served model name :", SERVED_MODEL_NAME)

EVAL_JSONL = download_eval_jsonl(EVAL_DATASET_ID)
print("eval jsonl        :", EVAL_JSONL)

## 6. Measure served NLL (same machinery as §F) + compare to reference

In [ ]:
import json


def load_examples(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def completion_token_split(tokenizer, prompt, completion):
    """prompt with special tokens, completion with add_special_tokens=False,
    concatenated. Same split as §E/§F. Returns int lists."""
    prompt_ids = tokenizer(prompt).input_ids
    completion_ids = tokenizer(completion, add_special_tokens=False).input_ids
    return prompt_ids, completion_ids, prompt_ids + completion_ids


def completion_logprobs_from_echo(logprobs_obj, input_ids, completion_start):
    if not isinstance(logprobs_obj, dict):
        raise RuntimeError(
            f"served 'logprobs' missing/not an object (got {type(logprobs_obj).__name__})")
    token_logprobs = logprobs_obj.get("token_logprobs")
    if token_logprobs is None:
        raise RuntimeError("served response has no logprobs.token_logprobs")
    if len(token_logprobs) < len(input_ids):
        raise RuntimeError(
            f"served logprobs length {len(token_logprobs)} < input length "
            f"{len(input_ids)}; cannot align completion span")
    comp = token_logprobs[completion_start:len(input_ids)]
    if any(lp is None for lp in comp):
        raise RuntimeError("served logprobs contain None inside the completion span")
    return comp


def _post_completion(url, headers, base_body, max_tokens):
    return requests.post(url, headers=headers, json={**base_body, "max_tokens": max_tokens}, timeout=120)


def served_completion_nll(endpoint, served_model_name, api_key, tokenizer, examples, max_tokens=0):
    """Teacher-forced completion-NLL via the served endpoint. Global sum of
    per-completion-token negative logprob / total completion tokens (matches
    §E _completion_nll). Falls back max_tokens 0 -> 1 if the server rejects 0."""
    url = endpoint.rstrip("/") + "/completions"
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    eff = max_tokens
    total_neg_logprob, total_tokens = 0.0, 0
    for i, ex in enumerate(examples):
        prompt_ids, completion_ids, input_ids = completion_token_split(
            tokenizer, ex["prompt"], ex["completion"])
        if not completion_ids:
            continue
        base_body = {"model": served_model_name, "prompt": input_ids,
                     "echo": True, "logprobs": 1, "temperature": 0}
        resp = _post_completion(url, headers, base_body, eff)
        if resp.status_code == 400 and eff == 0:
            eff = 1
            resp = _post_completion(url, headers, base_body, eff)
        resp.raise_for_status()
        choice = resp.json()["choices"][0]
        comp = completion_logprobs_from_echo(choice["logprobs"], input_ids, len(prompt_ids))
        total_neg_logprob += -float(sum(comp))
        total_tokens += len(comp)
        if (i + 1) % 10 == 0:
            print(f"[served] {i + 1} examples scored")
    if total_tokens == 0:
        raise RuntimeError("no completion tokens scored")
    return total_neg_logprob / total_tokens

In [ ]:
import math
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE)
examples = load_examples(EVAL_JSONL)
print(f"scoring {len(examples)} examples against {ENDPOINT} (model={SERVED_MODEL_NAME})")

served_nll = served_completion_nll(
    ENDPOINT, SERVED_MODEL_NAME, API_KEY, tokenizer, examples, max_tokens=MAX_TOKENS)

print("\n================== §G served LoRA-export summary ==================")
print(f"served (merged LoRA ISVC)     : {served_nll:.4f}  ({math.exp(served_nll):.4f} ppl)")
print(f"reference: hooks (context)    : {REFERENCE_HOOKS_NLL:.4f}  "
      f"(Δ = {served_nll - REFERENCE_HOOKS_NLL:+.4f} — includes the LoRA approximation)")

assert REFERENCE_NLL is not None, (
    "set REFERENCE_NLL to the local §E 'lora-export' NLL (ntk_lora_export_nll.ipynb) "
    "on this same adapter before running the gate")
rel = abs(served_nll - REFERENCE_NLL) / max(REFERENCE_NLL, 1e-9)
verdict = "PASS" if rel <= THRESHOLD else "FAIL"
print(f"reference: local lora-export  : {REFERENCE_NLL:.4f}  "
      f"(Δ = {served_nll - REFERENCE_NLL:+.4f}, relative = {rel * 100:.2f}%, "
      f"threshold {THRESHOLD * 100:.0f}%, {verdict})")
assert verdict == "PASS", (
    f"served merged-LoRA drifted {rel * 100:.2f}% from the local lora-export "
    f"(> {THRESHOLD * 100:.0f}%): the pod may not have loaded the merged weights correctly")
print("\nPASS — the served merged-LoRA reproduces the local export within tolerance.")

## Notes

- **What the gate proves.** served ≈ local `lora-export` (1%) is a *serving-correctness* check: the ISVC staged and loaded the merged weights and vLLM preserved the math. It is **not** an approximation test — that's the local §E `reference → lora-export` gap (~3%), surfaced here as the `hooks` context line.
- **Why merge instead of attach.** `lora_model_ids` is rejected on the vLLM LLM serving path (`_deploy_llm`), and the classical predictor route can't target an `llm` base — so a served LoRA number isn't reachable by attaching the adapter today. Merging is behaviourally identical (LoRA is linear: `W + (alpha/r)·BA`, and the exporter sets `alpha=r`). Swap in the commented `lora_model_ids` deploy once native attach ships.
- **Auth.** In-namespace runs need **no token** — Istio trusts same-namespace traffic and CogAPI scopes by `kubeflow-userid`. `API_KEY` is only for external-ingress/cross-namespace access.
- **No quantization.** Keep the merged ISVC unquantized so served (bf16/fp16) vs local (fp32) differ only by float noise (the 1% band).
- **`REFERENCE_NLL` must be the same adapter.** Compute it with `ntk_lora_export_nll.ipynb` (§E `lora-export` arm) on the exact adapter this notebook merges — otherwise the gate compares apples to oranges.
- **Cleanup.** The merged model is a full copy of the base weights; delete the `runs/_merged` dir and the ISVC/`llm` row when done to reclaim space.